# Продолжаем поиск и уточнение состава молодого семейства астероидов Emilkowalski

Данный ноутбук реализует последовательно выделение и проверки состава
семейства Emilkowalski на основе каталога собственных и оскулирующих элементов орбит (=собрала и привела в порядок все свои предыдущие вычисления).

Используемые методы:
- иерархическая кластеризация (HCM) с квази-метрикой d,
- метрика Холшевникова ρ₅,
- критерий KL,
- доп. проверка по метрике Холшевникова ρ₂ для оскулирующих орбит.

Цель — выделение устойчивого ядра семейства и проверка периферийных кандидатов.


In [1]:
import math
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster

# HCM cutoffs
D_CUT_RANGE = np.linspace(3, 10, 15)        # m/s
RHO5_CUT_RANGE = np.linspace(3e-4, 7e-4, 15)


PARENT_ID = "14627" # Родительское тело
N_CIRC = 3e4  # м/с, скорость, необходимая для движения по круговой орбите на радиусе Земли

SyntaxError: incomplete input (2505352884.py, line 22)

In [93]:
# ==========================================
# --Loading prop elements catalog (AstDys)--
# ==========================================

CATALOG_PATH = "/home/daria/crveniprojects/coursework/2025/proper_catalog24.dat"

#источник каталога собственных элементов - http://www2.boulder.swri.edu/~davidn/Proper24/proper_catalog24.dat

columns = [
    'a', 'a_er',
    'e', 'e_er',
    'sin_i', 'sin_i_er',
    'g_arcsec_per_year',
    's_arcsec_per_year',
    'H',
    'n_opps',
    'name',
    'number'
]

# Load as strings to preserve asteroid identifiers
proper = pd.read_csv(
    CATALOG_PATH,
    sep=r'\s+',
    header=None,
    names=columns,
    dtype=str
)

# Convert physical parameters to numeric values
numeric_cols = [
    'a', 'a_er',
    'e', 'e_er',
    'sin_i', 'sin_i_er',
    'g_arcsec_per_year',
    's_arcsec_per_year',
    'H'
]

proper[numeric_cols] = proper[numeric_cols].apply(
    pd.to_numeric,
    errors='coerce'
)

# ==========================================
# ---Loading osc elements catalog (AstDys-2)--
# ==========================================

OSC_PATH = "/home/daria/crveniprojects/coursework/catalogs/allnum25.cat"
osc = []
#источник каталога оскулирующих элементов - https://newton.spacedys.com/astdys2/index.php?pc=4

with open(OSC_PATH, 'r') as f:
    lines = f.readlines()

# Remove service and comment lines
data_lines = [
    line for line in lines
    if not line.startswith(("format", "rectype", "elem", "refsys", "END", "!"))
]

for line in data_lines:
    parts = line.strip().split()
    if len(parts) < 7:
        continue

    number = parts[0].strip("'")
    a = float(parts[2])
    e = float(parts[3])
    i = float(parts[4])
    Omega = float(parts[5])
    omega = float(parts[6])

    osc.append((number, a, e, i, Omega, omega))

osc_df = pd.DataFrame(
    osc,
    columns=['number', 'a_osc', 'e_osc', 'i_osc', 'Omega', 'omega']
)

# Ensure compatibility with proper elements catalog
osc_df['number'] = osc_df['number'].astype(str)

print(f"Loaded {len(proper)} objects from proper elements catalog")
print(f"Loaded {len(osc_df)} objects with osculating elements")


Loaded 1249051 objects from proper elements catalog
Loaded 793067 objects with osculating elements


In [56]:
# Выбор родительского тела
parent = proper.loc[proper["number"] == PARENT_ID].iloc[0]
print(f"Parent body: {parent['number']} ({parent['name']})")

# Фильтрация окрестности
def select_roi(df, parent, da=0.01, de=0.01, di=0.01):
    a0 = parent['a']
    e0 = parent['e']
    sin_i0 = parent['sin_i']
    
    roi = df[
        (df['a'] > a0 - da) & (df['a'] < a0 + da) &
        (df['e'] > e0 - de) & (df['e'] < e0 + de) &
        (df['sin_i'] > sin_i0 - di) & (df['sin_i'] < sin_i0 + di)
    ].copy()
    
    return roi

roi = select_roi(
    proper,
    parent,
    da=0.01,
    de=0.01,
    di=0.01
)

print(f"ROI size: {len(roi)} objects")


Parent body: 14627 (14627)
ROI size: 183 objects


In [57]:

emilkowalski_ids = [
    '14627', '126761', '224559', '256124', '434002', '476673','2015WH29',
    '2016CS377', '2022SA160', '2017UY114', '2014UV143',
    '2009VF107', '2018VB69'
]

print("Проверка наличия объектов семейства Emilkowalski в полном каталоге собственных элементов, а также в выделенной области roi.")
present_ids = []
roi_ids = []
missing_ids = []
missing_roi = []

# Проверка по str-сравнению
for aid in emilkowalski_ids:
    if (proper['number'] == aid).any():
        present_ids.append(aid)
    else:
        missing_ids.append(aid)

# Проверка области окрестности родительского тела
for id in emilkowalski_ids:
    if (roi['number'] == id).any():
        roi_ids.append(id)
    else:
        missing_roi.append(id)

print(f"Найдено в каталоге: {len(present_ids)} объектов")
if missing_ids:
    print(f"Не найдены в каталоге: {sorted(missing_ids)}")

print(f"Найдено в окрестности ROI - {len(roi_ids)} объектов: \n {sorted(roi_ids)}")
if missing_roi:
    print(f"Не найдены в окрестности ROI: {sorted(missing_roi)}")

📋 Проверка наличия объектов семейства Emilkowalski в полном каталоге собственных элементов, а также в выделенной области roi.
✅ Найдено в каталоге: 13 объектов
✅ Найдено в окрестности ROI - 13 объектов: 
 ['126761', '14627', '2009VF107', '2014UV143', '2015WH29', '2016CS377', '2017UY114', '2018VB69', '2022SA160', '224559', '256124', '434002', '476673']


In [75]:
def delta_v(u, v):
    a1, e1, s1 = u
    a2, e2, s2 = v
    a_mean = (a1 + a2) / 2
    da = (a1 - a2) / a_mean
    de = e1 - e2
    ds = s1 - s2
    AU = 1.496e8  # km
    mu = 1.327124e11  # km^3/s^2
    n = math.sqrt(mu / (a_mean * AU)**3)
    dv_km = n * a_mean * AU * math.sqrt((5/4) * da**2 + 2 * de**2 + 2 * ds**2)
    return dv_km * 1000  # m/s

def rho5(u, v):
    a1, e1, sin_i1 = u
    a2, e2, sin_i2 = v
    i1 = np.degrees(np.arcsin(sin_i1))
    i2 = np.degrees(np.arcsin(sin_i2))
    p1 = a1 * (1 - e1**2)
    p2 = a2 * (1 - e2**2)
    term = (1 + e1**2) * p1 + (1 + e2**2) * p2
    term -= 2 * np.sqrt(p1 * p2) * (e1 * e2 + np.cos(np.radians(i1 - i2)))
    return np.sqrt(term)

def run_hcm(objects, metric_func, cutoff):
    if metric_func == d_metric:
        data = objects[['a', 'e', 'sin_i']].to_numpy()
        dist = pdist(data, metric=delta_v)
    else metric_func == rho5_metric:
        data = objects[['a', 'e', 'sin_i']].to_numpy()
        dist = pdist(data, metric=rho5)
    
    Z = linkage(dist, method='single')
    return fcluster(Z, t=cutoff, criterion='distance')

# Запуск кластеризации по диапазонам порогов
print("Кластеризация по метрике d")
clusters_d = []
for dcut in D_CUT_RANGE:
    labels = run_hcm(roi, d_metric, dcut)
    clusters_d.append(labels)
    n_clusters = len(np.unique(labels))
    main_cluster_size = np.bincount(labels).max()
    print(f"  Порог d = {dcut:5.2f} м/с → кластеров: {n_clusters:3d}, крупнейший: {main_cluster_size:3d} объектов")

print("\nКластеризация по метрике Холшевникова ρ₅")
clusters_rho5 = []
for rcut in RHO5_CUT_RANGE:
    labels = run_hcm(roi, rho5_metric, rcut)
    clusters_rho5.append(labels)
    n_clusters = len(np.unique(labels))
    main_cluster_size = np.bincount(labels).max()
    print(f"  Порог ρ₅ = {rcut:.5f} → кластеров: {n_clusters:3d}, крупнейший: {main_cluster_size:3d} объектов")

=== Кластеризация по метрике d (Zappalà) ===
  Порог d =  3.00 м/с → кластеров: 169, крупнейший:  14 объектов
  Порог d =  3.50 м/с → кластеров: 168, крупнейший:  16 объектов
  Порог d =  4.00 м/с → кластеров: 167, крупнейший:  16 объектов
  Порог d =  4.50 м/с → кластеров: 166, крупнейший:  17 объектов
  Порог d =  5.00 м/с → кластеров: 166, крупнейший:  17 объектов
  Порог d =  5.50 м/с → кластеров: 166, крупнейший:  17 объектов
  Порог d =  6.00 м/с → кластеров: 166, крупнейший:  17 объектов
  Порог d =  6.50 м/с → кластеров: 166, крупнейший:  17 объектов
  Порог d =  7.00 м/с → кластеров: 166, крупнейший:  17 объектов
  Порог d =  7.50 м/с → кластеров: 166, крупнейший:  17 объектов
  Порог d =  8.00 м/с → кластеров: 166, крупнейший:  17 объектов
  Порог d =  8.50 м/с → кластеров: 166, крупнейший:  17 объектов
  Порог d =  9.00 м/с → кластеров: 166, крупнейший:  17 объектов
  Порог d =  9.50 м/с → кластеров: 166, крупнейший:  17 объектов
  Порог d = 10.00 м/с → кластеров: 166, крупн

In [76]:
def find_stable_core(clusters_d, clusters_rho5, parent_id, roi_df):
    # Найти индекс ближайшего порога к 5.0 м/с в D_CUT_RANGE
    d_idx = np.argmin(np.abs(D_CUT_RANGE - 5.0))
    # Найти индекс ближайшего порога к 0.0005 в RHO5_CUT_RANGE
    rho5_idx = np.argmin(np.abs(RHO5_CUT_RANGE - 0.0005))
    
    # Получить метки кластеров для этих порогов
    cl_d = clusters_d[d_idx]      # массив длины len(roi)
    cl_rho5 = clusters_rho5[rho5_idx]
    
    # Найти позицию родительского тела в roi
    parent_row = roi_df[roi_df['number'] == parent_id].iloc[0]
    parent_index_in_roi = roi_df.index.get_loc(parent_row.name)
    
    # Определить метку кластера, содержащего родителя
    parent_cluster_d = cl_d[parent_index_in_roi]
    parent_cluster_rho5 = cl_rho5[parent_index_in_roi]
    
    # Маски: какие объекты входят в кластер с родителем
    mask_d = (cl_d == parent_cluster_d)
    mask_rho5 = (cl_rho5 == parent_cluster_rho5)
    
    # Ядро = пересечение
    core_mask = mask_d & mask_rho5
    core_numbers = roi_df[core_mask]['number'].tolist()
    
    return core_numbers

core_members = find_stable_core(clusters_d, clusters_rho5, PARENT_ID, roi)
print("Ядро семейства Emilkowalski включает", len(core_members), 'объектов:', sorted(core_members))

Ядро семейства Emilkowalski включает 17 объектов: ['126761', '14627', '2009UL13', '2009VF107', '2014UV143', '2015WH29', '2016CS377', '2017UY114', '2018VB69', '2019VE44', '2020UZ20', '2021TU55', '2022SA160', '224559', '256124', '434002', '476673']


In [78]:
# Сравнение с известным списком членов семейства
core_set = set(core_members)
known_set = set(emilkowalski_ids)

true_positives = core_set & known_set          # Найдены и в ядре, и в списке
false_negatives = known_set - core_set        # Есть в списке, но не вошли в ядро
false_positives = core_set - known_set        # Вошли в ядро, но отсутствуют в списке

print("\nСравнение с известным составом семейства Emilkowalski:")
print(f"Совпадений (TP): {len(true_positives)} — {sorted(true_positives)}")
if false_negatives:
    print(f"Пропущено (FN): {len(false_negatives)} — {sorted(false_negatives)}")
if false_positives:
    print(f"Новые кандидаты (FP): {len(false_positives)} — {sorted(false_positives)}")

print(f"\nТочность (Precision): {len(true_positives) / len(core_set):.2%}")
print(f"Полнота (Recall):    {len(true_positives) / len(known_set):.2%}")


Сравнение с известным составом семейства Emilkowalski:
Совпадений (TP): 13 — ['126761', '14627', '2009VF107', '2014UV143', '2015WH29', '2016CS377', '2017UY114', '2018VB69', '2022SA160', '224559', '256124', '434002', '476673']
⚠️ Новые кандидаты (FP): 4 — ['2009UL13', '2019VE44', '2020UZ20', '2021TU55']

Точность (Precision): 76.47%
Полнота (Recall):    100.00%


In [79]:
# Вспомогательные функции для KL-критерия

def sin_i_to_i_deg(sin_i_value):
    v = float(sin_i_value)
    v = max(min(v, 1.0), -1.0)
    return math.degrees(math.asin(v))

def uv_from_elements(a, e, i_deg, omega_deg=0.0, Omega_deg=0.0):
    a = float(a); e = float(e); i_deg = float(i_deg)
    i_rad = np.radians(i_deg)
    omega_rad = np.radians(float(omega_deg))
    Omega_rad = np.radians(float(Omega_deg))
    p = a * (1.0 - e * e)
    if p <= 0:
        p = max(abs(p), 1e-12)
    sqrt_p = np.sqrt(p)
    Rz_Omega = np.array([[np.cos(Omega_rad), -np.sin(Omega_rad), 0],
                         [np.sin(Omega_rad),  np.cos(Omega_rad), 0],
                         [0, 0, 1]])
    Rx_i = np.array([[1, 0, 0],
                     [0, np.cos(i_rad), -np.sin(i_rad)],
                     [0, np.sin(i_rad),  np.cos(i_rad)]])
    Rz_omega = np.array([[np.cos(omega_rad), -np.sin(omega_rad), 0],
                         [np.sin(omega_rad),  np.cos(omega_rad), 0],
                         [0, 0, 1]])
    R = Rz_Omega @ Rx_i @ Rz_omega
    u = sqrt_p * R @ np.array([1.0, 0.0, 0.0])
    v = sqrt_p * R @ np.array([0.0, 1.0, 0.0])
    return u, v

def rho2_from_uv(u1, v1, u2, v2):
    du = u1 - u2
    dv = v1 - v2
    return float(np.sqrt(np.dot(du, du) + np.dot(dv, dv)))

def rho5_min_numeric(elemA, elemB, grid_N=36):
    a1, e1, i1 = float(elemA['a']), float(elemA['e']), float(elemA['i'])
    a2, e2, i2 = float(elemB['a']), float(elemB['e']), float(elemB['i'])
    u1, v1 = uv_from_elements(a1, e1, i1, omega_deg=0.0, Omega_deg=0.0)
    
    omegas = np.linspace(0.0, 360.0, grid_N, endpoint=False)
    Omegas = np.linspace(0.0, 360.0, grid_N, endpoint=False)
    Om_mesh, om_mesh = np.meshgrid(Omegas, omegas, indexing='xy')
    om_flat = om_mesh.ravel()
    Om_flat = Om_mesh.ravel()
    
    rhos = np.empty_like(om_flat, dtype=float)
    for k, (om_deg, Om_deg) in enumerate(zip(om_flat, Om_flat)):
        u2, v2 = uv_from_elements(a2, e2, i2, omega_deg=om_deg, Omega_deg=Om_deg)
        rhos[k] = rho2_from_uv(u1, v1, u2, v2)
    
    idx_min = int(np.argmin(rhos))
    return float(rhos[idx_min])

In [80]:
def compute_K_values(core_numbers, roi_df):
    roi_with_i = roi_df[roi_df['number'].isin(core_numbers)].copy()
    roi_with_i['i'] = roi_with_i['sin_i'].apply(sin_i_to_i_deg)
    records = roi_with_i.to_dict('records')
    n = len(records)
    if n < 2:
        return np.nan, np.nan

    main_id = '14627'
    pairs = list(combinations(range(n), 2))
    rho_vals = []

    for i, j in pairs:
        row_i, row_j = records[i], records[j]
        rho = rho5_min_numeric(row_i, row_j, grid_N=36)
        rho_vals.append((row_i['number'], row_j['number'], rho))

    rho_df = pd.DataFrame(rho_vals, columns=['i_num', 'j_num', 'rho'])
    K2 = rho_df['rho'].max()
    mask_main = (rho_df['i_num'] == main_id) | (rho_df['j_num'] == main_id)
    K1 = rho_df.loc[mask_main, 'rho'].max() if mask_main.any() else np.nan
    return K1, K2

def apply_KL(core_numbers, roi_df, K1, K2):
    roi_with_i = roi_df.copy()
    roi_with_i['i'] = roi_with_i['sin_i'].apply(sin_i_to_i_deg)
    core_records = roi_with_i[roi_with_i['number'].isin(core_numbers)].to_dict('records')
    all_records = roi_with_i.to_dict('records')
    main_record = next(r for r in core_records if r['number'] == '14627')
    
    candidates = []
    for cand in all_records:
        L1 = rho5_min_numeric(cand, main_record, grid_N=36)
        L2_vals = [rho5_min_numeric(cand, member, grid_N=36) for member in core_records]
        L2 = max(L2_vals) if L2_vals else np.nan
        in_family = (L1 <= K1) or (L2 <= K2)
        candidates.append({
            'number': cand['number'],
            'L1': L1,
            'L2': L2,
            'in_family': in_family
        })
    
    kl_df = pd.DataFrame(candidates)
    return kl_df[kl_df['in_family']]['number'].tolist()

In [83]:
K1, K2 = compute_K_values(core_members, roi)
kl_candidates = apply_KL(core_members, roi, K1, K2)

print(f"K1 = {K1:.6f}, K2 = {K2:.6f}")
print("Прошли отбор по критерию KL", len(kl_candidates), 'астероидов', sorted(kl_candidates))

K1 = 0.000718, K2 = 0.000843
Прошли отбор по критерию KL 18 астероидов ['126761', '14627', '2009UL13', '2009VF107', '2014UV143', '2014WE584', '2015WH29', '2016CS377', '2017UY114', '2018VB69', '2019VE44', '2020UZ20', '2021TU55', '2022SA160', '224559', '256124', '434002', '476673']


In [84]:
kl_set = set(kl_candidates)
known_set = set(emilkowalski_ids)

true_positives = kl_set & known_set          # Найдены и в K/L, и в списке
false_negatives = known_set - kl_set       # Есть в списке, но не вошли в K/L
false_positives = kl_set - known_set        # Вошли в K/L, но отсутствуют в списке

print("\nСравнение с известным составом семейства Emilkowalski:")
print(f"Совпадений (TP): {len(true_positives)} — {sorted(true_positives)}")
if false_negatives:
    print(f"Пропущено (FN): {len(false_negatives)} — {sorted(false_negatives)}")
if false_positives:
    print(f"Новые кандидаты (FP): {len(false_positives)} — {sorted(false_positives)}")

print(f"\nТочность (Precision): {len(true_positives) / len(kl_set):.2%}")
print(f"Полнота (Recall):    {len(true_positives) / len(known_set):.2%}")


Сравнение с известным составом семейства Emilkowalski:
Совпадений (TP): 13 — ['126761', '14627', '2009VF107', '2014UV143', '2015WH29', '2016CS377', '2017UY114', '2018VB69', '2022SA160', '224559', '256124', '434002', '476673']
⚠️ Новые кандидаты (FP): 5 — ['2009UL13', '2014WE584', '2019VE44', '2020UZ20', '2021TU55']

Точность (Precision): 72.22%
Полнота (Recall):    100.00%


-----------------------------------------------------------------------------------------
Новые кандидаты (HCM-метод): 4 астероида — '2009UL13', '2019VE44', '2020UZ20', '2021TU55'

Новые кандидаты (K/L-метод): 5 астероидов — '2009UL13', '2019VE44', '2020UZ20', '2021TU55', '2014WE584'

In [96]:
# Дополнительная проверка по rho2
def compute_rho2(core_numbers, roi_df, osc_df):
    # Сопоставление ядра с оскулирующими элементами
    core_roi = roi_df[roi_df['number'].isin(core_numbers)]
    merged = core_roi.merge(osc_df, on='number', how='inner')
    
    if len(merged) == 0:
        print("Нет пересечения ядра с каталогом оскулирующих элементов.")
        return []
    
    # Подготовка данных
    data_rho2 = merged[['a_osc', 'e_osc', 'i_osc', 'Omega', 'omega']].to_numpy()
    dist_rho2 = pdist(data_rho2, metric=rho2)
    Z_rho2 = linkage(dist_rho2, method='single')
    
    # Кластеризация
    rho2_cutoff = 0.08
    labels = fcluster(Z_rho2, t=rho2_cutoff, criterion='distance')
    merged['cluster_rho2'] = labels
    
    # Найти кластер с родителем
    if PARENT_ID not in merged['number'].values:
        print(f"Родительское тело {PARENT_ID} отсутствует в оскулирующем каталоге.")
        return []
    
    parent_cluster = merged.loc[merged['number'] == PARENT_ID, 'cluster_rho2'].iloc[0]
    rho2_cluster = merged[merged['cluster_rho2'] == parent_cluster]
    
    return rho2_cluster['number'].tolist()

In [87]:
# --- Вывод результатов по метрике ρ₂ ---
rho2_results = compute_rho2(core_members, roi, osc_df)
print("Астероиды, вошедшие в кластер по метрике ρ₂:", sorted(rho2_results))

# --- Комментарий о составе оскулирующего каталога ---
present_in_osc = [aid for aid in emilkowalski_ids if aid in osc_df['number'].values]
missing_in_osc = [aid for aid in emilkowalski_ids if aid not in osc_df['number'].values]

print(f"\nОскулирующий каталог содержит только {len(present_in_osc)} из {len(emilkowalski_ids)} известных членов семейства Emilkowalski:")
print(f"Присутствуют: {sorted(present_in_osc)}")
if missing_in_osc:
    print(f"Отсутствуют: {sorted(missing_in_osc)}")
print("Поэтому кластеризация по ρ₂ ограничена только пронумерованными объектами.")

Астероиды, вошедшие в кластер по метрике ρ₂: ['126761', '14627', '224559', '256124', '434002', '476673']

Оскулирующий каталог содержит только 6 из 13 известных членов семейства Emilkowalski:
Присутствуют: ['126761', '14627', '224559', '256124', '434002', '476673']
Отсутствуют: ['2009VF107', '2014UV143', '2015WH29', '2016CS377', '2017UY114', '2018VB69', '2022SA160']
Поэтому кластеризация по ρ₂ ограничена только пронумерованными объектами.


In [97]:
# Итоговый результат
def build_family_table(core, kl, rho2, known_ids, roi_df, osc_df):
    """
    Собирает итоговую таблицу состава семейства Emilkowalski.
    """
    all_ids = set(core) | set(kl) | set(rho2) | set(known_ids)
    table = []

    # Множество объектов, имеющихся в оскулирующем каталоге
    in_osc_set = set(osc_df['number'].values)

    for aid in sorted(all_ids):
        row = {
            'number': aid,
            'in_known': aid in known_ids,
            'in_core': aid in core,
            'in_KL': aid in kl,
            'a': None,
            'e': None,
            'sin_i': None
        }
        
        # Определяем статус по rho_2
        if aid in in_osc_set:
            row['in_rho2'] = aid in rho2
        else:
            row['in_rho2'] = 'Not in osc'
        
        # Добавляем орбитальные элементы из ROI
        match = roi_df[roi_df['number'] == aid]
        if not match.empty:
            row['a'] = match.iloc[0]['a']
            row['e'] = match.iloc[0]['e']
            row['sin_i'] = match.iloc[0]['sin_i']
        
        table.append(row)
    
    df = pd.DataFrame(table)
    # Сортировка: сначала известные члены
    df['known_order'] = df['number'].apply(lambda x: known_ids.index(x) if x in known_ids else 999)
    df = df.sort_values(['known_order', 'number']).drop(columns=['known_order'])
    return df

In [98]:
# === ФИНАЛЬНЫЙ АНАЛИЗ СОСТАВА СЕМЕЙСТВА EMILKOWALSKI ===

# 1. Сравнение ядра с известным составом
core_set = set(core_members)
known_set = set(emilkowalski_ids)
tp = core_set & known_set          # Правильные положительные
fn = known_set - core_set         # Пропущенные (ложные отрицательные)
fp = core_set - known_set         # Новые кандидаты (ложные положительные)

print("СРАВНЕНИЕ ЯДРА С ИЗВЕСТНЫМ СОСТАВОМ:")
print(f"✅ Совпадений (TP): {len(tp)} — {sorted(tp)}")
if fn:
    print(f"❌ Пропущено (FN): {len(fn)} — {sorted(fn)}")
if fp:
    print(f"⚠️  Новые кандидаты (FP): {len(fp)} — {sorted(fp)}")

print(f"\nТочность (Precision): {len(tp) / len(core_set):.2%}")
print(f"Полнота (Recall):    {len(tp) / len(known_set):.2%}")

# 2. Вывод по KL-критерию
print(f"\nКРИТЕРИЙ KL:")
print(f"   K1 = {K1:.6f}, K2 = {K2:.6f}")
print(f"   Отобрано объектов: {len(kl_candidates)}")

# 3. Вывод по ρ₂
present_in_osc = [aid for aid in emilkowalski_ids if aid in osc_df['number'].values]
missing_in_osc = [aid for aid in emilkowalski_ids if aid not in osc_df['number'].values]

print(f"\nМЕТРИКА \rho_2 (оскулирующий каталог):")
print(f"   В кластере: {len(rho2_results)} объектов — {sorted(rho2_results)}")
print(f"   Оскулирующий каталог содержит только {len(present_in_osc)} из {len(emilkowalski_ids)} известных членов.")
if missing_in_osc:
    print(f"   Отсутствуют: {sorted(missing_in_osc)}")

# 4. Итоговая таблица
print("\n" + "="*80)
print("📋 ИТОГОВАЯ ТАБЛИЦА СОСТАВА СЕМЕЙСТВА EMILKOWALSKI")
print("="*80)

final_table = build_family_table(
    core=core_members,
    kl=kl_candidates,
    rho2=rho2_results,
    known_ids=emilkowalski_ids,
    roi_df=roi,
    osc_df=osc_df  
)

# Красивый вывод таблицы
formatters = {
    'a': '{:.5f}'.format,
    'e': '{:.5f}'.format,
    'sin_i': '{:.5f}'.format
}
print(final_table.to_string(
    index=False,
    formatters=formatters,
    columns=['number', 'in_known', 'in_core', 'in_KL', 'in_rho2', 'a', 'e', 'sin_i']
))

СРАВНЕНИЕ ЯДРА С ИЗВЕСТНЫМ СОСТАВОМ:
✅ Совпадений (TP): 13 — ['126761', '14627', '2009VF107', '2014UV143', '2015WH29', '2016CS377', '2017UY114', '2018VB69', '2022SA160', '224559', '256124', '434002', '476673']
⚠️  Новые кандидаты (FP): 4 — ['2009UL13', '2019VE44', '2020UZ20', '2021TU55']

Точность (Precision): 76.47%
Полнота (Recall):    100.00%

КРИТЕРИЙ KL:
   K1 = 0.000718, K2 = 0.000843
   Отобрано объектов: 18

МЕТРИКА ρ₂ (оскулирующий каталог):
   В кластере: 6 объектов — ['126761', '14627', '224559', '256124', '434002', '476673']
   Оскулирующий каталог содержит только 6 из 13 известных членов.
   Отсутствуют: ['2009VF107', '2014UV143', '2015WH29', '2016CS377', '2017UY114', '2018VB69', '2022SA160']

📋 ИТОГОВАЯ ТАБЛИЦА СОСТАВА СЕМЕЙСТВА EMILKOWALSKI
   number  in_known  in_core  in_KL    in_rho2       a       e   sin_i
    14627      True     True   True       True 2.59929 0.17461 0.29577
   126761      True     True   True       True 2.59955 0.17428 0.29579
   224559      True  